# NER — service eval notebook

Hits the deployed `/ner` endpoint with a caller-supplied schema and shows the full extraction result per test case.

Each case carries its own inline schema so different cases can exercise different entity/relation types.

Scoring is **structural**: an expected entity matches if the type is correct and the name contains or is contained in the expected name (case-insensitive). An expected relation matches if the type is correct and the head/tail entity types match. Attribute checks are advisory — mismatches are flagged but do not fail the case.

Test cases live in `eval/fixtures/ner_cases.json`.

In [ ]:
import os, json, requests
from pathlib import Path

NLP_BASE_URL = os.environ.get('NLP_BASE_URL', 'https://wiig.dia.fi.upm.es/nlp').rstrip('/')
ENDPOINT     = f'{NLP_BASE_URL}/ner'
HEADERS      = {'Content-Type': 'application/json', 'X-API-Key': f'{os.environ.get("NLP_API_KEY")}'}
TIMEOUT      = 120

CASES = json.loads(Path('fixtures/ner_cases.json').read_text(encoding='utf-8'))
print(f'{len(CASES)} cases -> {ENDPOINT}')

In [ ]:
def call_ner(inp):
    r = requests.post(ENDPOINT, json=inp, headers=HEADERS, timeout=TIMEOUT)
    r.raise_for_status()
    return r.json()


def render(resp, expected):
    entities  = resp.get('entities',  [])
    relations = resp.get('relations', [])

    print('  ENTITIES')
    if not entities:
        print('    (none)')
    for i, e in enumerate(entities):
        sub   = f"  subtype={e['subtype']!r}"  if e.get('subtype')  else ''
        attrs = f"  attrs={e['attributes']}"   if e.get('attributes') else ''
        conf  = f"  conf={e['confidence']:.2f}"
        print(f'    [{i}] type={e["type"]!r}  name={e["name"]!r}{sub}{attrs}{conf}')

    print('  RELATIONS')
    if not relations:
        print('    (none)')
    for r in relations:
        sub   = f"  subtype={r['subtype']!r}" if r.get('subtype') else ''
        attrs = f"  attrs={r['attributes']}"  if r.get('attributes') else ''
        conf  = f"  conf={r['confidence']:.2f}"
        h_name = entities[r['head']]['name'] if 0 <= r['head'] < len(entities) else '?'
        t_name = entities[r['tail']]['name'] if 0 <= r['tail'] < len(entities) else '?'
        print(f'    {h_name!r} --[{r["type"]}]--> {t_name!r}{sub}{attrs}{conf}')

    # attribute advisory checks
    for exp_e in expected.get('entities', []):
        exp_attrs = exp_e.get('attributes', {})
        if not exp_attrs:
            continue
        matched_e = next((e for e in entities if _entity_matches(exp_e, e)), None)
        if matched_e:
            got = matched_e.get('attributes', {})
            for k, v in exp_attrs.items():
                got_v = got.get(k)
                icon  = '✅' if str(got_v).lower() == str(v).lower() else '⚠️ '
                print(f'  {icon} attr {k}: expected={v!r}  got={got_v!r}')

    for exp_r in expected.get('relations', []):
        exp_attrs = exp_r.get('attributes', {})
        if not exp_attrs:
            continue
        matched_r = next((r for r in relations if _relation_matches(exp_r, r, entities)), None)
        if matched_r:
            got = matched_r.get('attributes', {})
            for k, v in exp_attrs.items():
                try:
                    got_v = got.get(k)
                    icon  = '✅' if float(got_v) == float(v) else '⚠️ '
                except (TypeError, ValueError):
                    icon  = '⚠️ '
                    got_v = got.get(k)
                print(f'  {icon} attr {k}: expected={v!r}  got={got_v!r}')

In [ ]:
def _name_matches(exp_name, got_name):
    a, b = exp_name.lower(), got_name.lower()
    return a in b or b in a


def _entity_matches(exp, entity):
    if entity['type'] != exp['type']:
        return False
    if 'name' in exp and not _name_matches(exp['name'], entity['name']):
        return False
    if 'subtype' in exp and entity.get('subtype') != exp['subtype']:
        return False
    return True


def _relation_matches(exp, rel, entities):
    if rel['type'] != exp['type']:
        return False
    h = entities[rel['head']] if 0 <= rel['head'] < len(entities) else None
    t = entities[rel['tail']] if 0 <= rel['tail'] < len(entities) else None
    if 'head_type' in exp and (h is None or h['type'] != exp['head_type']):
        return False
    if 'tail_type' in exp and (t is None or t['type'] != exp['tail_type']):
        return False
    return True


def score(expected, resp):
    entities  = resp.get('entities',  [])
    relations = resp.get('relations', [])

    exp_entities  = expected.get('entities',  [])
    exp_relations = expected.get('relations', [])

    em = sum(1 for e in exp_entities  if any(_entity_matches(e, got) for got in entities))
    rm = sum(1 for r in exp_relations if any(_relation_matches(r, got, entities) for got in relations))

    total_exp = len(exp_entities) + len(exp_relations)
    total_got = em + rm

    # false-positive penalty: if no entities expected, flag any entities extracted
    fp = len(entities) if not exp_entities and not exp_relations else 0

    return total_got, total_exp, fp

In [ ]:
results = []
for case in CASES:
    print(f"── {case['id']}: {case['description']}")
    try:
        resp = call_ner(case['input'])
    except Exception as exc:
        print('   ERROR:', exc); print(); results.append((case['id'], 0, 1, 0)); continue

    expected = case.get('expected', {})
    render(resp, expected)

    m, total, fp = score(expected, resp)
    if total == 0 and fp == 0:
        icon = '✅'
    elif total == 0 and fp > 0:
        icon = '❌'
    else:
        icon = '✅' if m == total and fp == 0 else '❌'
    fp_note = f'  ({fp} false-positive entities)' if fp else ''
    print(f'   {icon} matched {m}/{total}{fp_note}')
    if case.get('expected', {}).get('notes'):
        print(f'   note: {case["expected"]["notes"]}')
    print()
    results.append((case['id'], m, total, fp))

In [ ]:
passed = sum(1 for _, m, t, fp in results if m == t and fp == 0)
tm = sum(m  for _, m, _, _  in results)
tt = sum(t  for _, _, t, _  in results)
print('SCORECARD · /ner')
print(f'  Cases passing   : {passed}/{len(results)}')
print(f'  Items matched   : {tm}/{tt}')
for cid, m, t, fp in results:
    fp_note = f'  +{fp} FP' if fp else ''
    print(f"   {'✅' if m==t and fp==0 else '❌'} {cid}: {m}/{t}{fp_note}")